In [ ]:
%%bash
set -eo pipefail

root="$PWD"
work="$root/archives/cryo-benchmark"
mkdir -p "$work"

export PATH="$work/bin:$PATH"
if ! command -v uv >/dev/null; then
    curl -fsSL https://astral.sh/uv/install.sh | UV_UNMANAGED_INSTALL="$work/bin" sh
fi
export MPLBACKEND=Agg TORCH_HOME="$work/torch-cache"
cd "$work"

repo="$work/ModelAngelo"
if [ ! -d "$repo" ]; then
    git clone https://github.com/3dem/model-angelo.git "$repo"
fi
git -C "$repo" checkout --quiet --detach 994945b

env="$work/ModelAngelo-env"
index=(--index-url https://download.pytorch.org/whl/cu126)
if [ "$(uname -s)" = Darwin ]; then index=(); fi
if [ ! -x "$env/bin/python" ]; then uv venv --python 3.11 "$env"; fi
uv pip install --python "$env/bin/python" "${index[@]}" torch==2.9.1 torchvision==0.24.1
uv pip install --python "$env/bin/python" "$repo"
"$env/bin/model_angelo" setup_weights --bundle-name nucleotides


In [ ]:
%%bash
set -eo pipefail

root="$PWD"
work="$root/archives/cryo-benchmark"

method=ModelAngelo
device=0
if [ "$(uname -s)" = Darwin ]; then device=cpu; fi
export MPLBACKEND=Agg TORCH_HOME="$work/torch-cache"

for target in 22XP 32MC; do
    data="$root/data/targets/$target"
    map=("$data"/inputs/emd_*.map)
    fasta="$data/inputs/rcsb_pdb_${target}.fasta"
    out="$work/runs/$method/$target"

    time "$work/ModelAngelo-env/bin/model_angelo" build -v "${map[0]}" -pf "$fasta" -o "$out" -d "$device"
done


In [ ]:
%%bash
set -eo pipefail

root="$PWD"
work="$root/archives/cryo-benchmark"

if [ "$(uname -s)" = Linux ] && ! command -v g++ >/dev/null; then
    apt-get update
    apt-get install -y g++
fi

phenix_env="${PHENIX_ENV:-$work/phenix/phenix_env.sh}"
if [ ! -f "$phenix_env" ]; then
    for script in "$root/../cryo-benchmark-work/phenix/phenix_env.sh" "$HOME"/phenix-*/phenix_env.sh /Applications/phenix-*/phenix_env.sh; do
        if [ -f "$script" ]; then phenix_env="$script"; break; fi
    done
fi

method="ModelAngelo"
if [ ! -d "$work/USalign" ]; then
    git clone https://github.com/pylelab/USalign.git "$work/USalign"
fi
make -C "$work/USalign" USalign

for target in 22XP 32MC; do
    out="$work/runs/$method/$target"
    reference="$root/data/targets/$target/reference/$target.cif"
    model="$out/$target.cif"
    test -s "$model"
    "$work/USalign/USalign" "$model" "$reference" -mol prot -mm 1 -ter 1 -outfmt 2 > "$out/usalign.tsv"
    if [ -f "$phenix_env" ]; then
        (source "$phenix_env"; phenix.chain_comparison "$reference" "$model" \
            max_dist=3 unique_part_of_target_only=False test_unique_part_of_target_only=False \
            use_crystal_symmetry=False) > "$out/chain-comparison.txt"
    fi
done
